# AIgnition — Train the Bayesian forecasting model (Colab)

Colab runs Linux with a working C/C++ toolchain, so PyMC compiles and samples normally here (unlike Windows without a configured compiler).

**Steps:** clone the repo → install training deps → provide data → fit the Bayesian model → verify the scored pipeline → download `model.pkl`.

Run the cells top to bottom.

## 1. Clone the repository
Set `REPO_URL` to your public GitHub repo (after you push it).

In [ ]:
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"  # <-- edit me
import os
repo_name = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
if not os.path.isdir(repo_name):
    !git clone $REPO_URL
%cd $repo_name
!ls

## 2. Install training dependencies
Installs PyMC/ArviZ plus the scored-path deps and the local package.

In [ ]:
!pip install -q -r requirements-train.txt
!pip install -q -e .
import pymc; print('pymc', pymc.__version__)

## 3. Provide the data
Set `USE_SAMPLE = True` to train on the committed synthetic sample, or `False` to upload the real GA4 + Shopify CSVs into `data/`.

In [ ]:
USE_SAMPLE = True

if USE_SAMPLE:
    DATA_DIR = 'data/sample'
else:
    DATA_DIR = 'data'
    import os, glob
    os.makedirs(DATA_DIR, exist_ok=True)
    for f in glob.glob(os.path.join(DATA_DIR, '*.csv')):
        os.remove(f)
    from google.colab import files
    uploaded = files.upload()  # select your GA4 + Shopify CSVs
    for name in uploaded:
        os.replace(name, os.path.join(DATA_DIR, name))
print('Using DATA_DIR =', DATA_DIR)
!ls $DATA_DIR

## 4. Fit the hierarchical Bayesian model
Writes the compiled posterior to `pickle/model.pkl`. Increase `--draws` for a higher-quality fit.

In [ ]:
!python train.py --data-dir $DATA_DIR --out pickle/model.pkl --method bayesian --draws 1000

## 5. Verify the scored pipeline against the new model
This is the exact flow the judges run — it must produce a valid `predictions.csv` offline.

In [ ]:
!bash run.sh $DATA_DIR pickle/model.pkl output/predictions.csv
import pandas as pd
df = pd.read_csv('output/predictions.csv')
assert list(df.columns) == ['level','entity','horizon_days','metric','p10','p50','p90']
assert set(df['horizon_days']) == {30, 60, 90}
print('OK:', len(df), 'rows')
df.head(12)

## 6. Download the trained model
Then commit it to the repo: `git add pickle/model.pkl && git commit -m "feat: Bayesian-fitted model"`.

In [ ]:
from google.colab import files
files.download('pickle/model.pkl')